# 第46章 累积分布图（ecdfplot）

用ECDF直接展示小于等于某值的样本比例，无需选择分箱或带宽。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

比较分位数、阈值覆盖率或不同组的完整累计分布。

## 数据结构

一列连续数值，可按类别分组。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 complementary=False 改为 complementary=True，对比累计分布与互补累计分布的曲线方向
2. 修改 stat="proportion" 为 stat="count"，观察比例与计数的纵轴差异
3. 在图上添加 axvline 标记特定分位数（如中位数位置），说明ECDF在分位数读取中的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


sns.set_theme(style="whitegrid", context="notebook")
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    category = diamonds["cut"], channel=diamonds["color"], region=diamonds["clarity"],
    order_value = diamonds["price"], items=diamonds["carat"],
    satisfied=np.where(diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"),
)
orders = orders_full.sample(2_000, random_state=36).copy()
taxis = pd.read_csv(f"{base_url}/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
marketing_full = taxis.assign(
    channel = taxis["payment"].fillna("unknown"), visits=taxis["distance"],
    ad_spend = taxis["tip"], sales=taxis["total"],
    conversion = (taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(min(2_000, len(marketing_full)), random_state=36).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
daily = flights.assign(
    date = pd.to_datetime(flights["year"].astype("string") + "-" + flights["month"] + "-01"),
    region = "AirPassengers", sales=flights["passengers"],
)
print(f"Diamonds：{len(diamonds):,} 行；NYC Taxis：{len(taxis):,} 行；Flights：{len(flights):,} 行")
print("图表兼容列均由公开数据原始字段直接映射；高成本图使用固定 2,000 行样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.3))
sns.ecdfplot(data=orders, x="order_value", color="#1a73e8", ax=ax)
ax.axhline(0.5, color="#9aa0a6", linestyle="--")
ax.set(title="订单金额累计分布", xlabel="客单价（元）", ylabel="累计比例")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.ecdfplot(data=orders, x="order_value", hue="category", palette="colorblind", ax=ax)
ax.axvline(300, color="#d93025", linestyle="--", label="300元阈值")
ax.set(title="品类客单价累计分布", xlabel="客单价（元）", ylabel="累计比例")
fig.tight_layout()
plt.show()


## 3. 参数说明

- stat：proportion/count
- complementary：互补累计
- hue：分组
- weights：权重


## 4. 结果解读

在任意X值读取累计比例，或在给定比例处读取分位值。


## 常见误区

- 不理解阶梯线含义
- 组间样本量不等时比较count
- 把陡峭部分解释为时间变化


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.ecdfplot(data=marketing, x="conversion", hue="channel", complementary=True, palette="colorblind", ax=ax)
ax.set(title="转化率超过阈值的比例", xlabel="转化率阈值", ylabel="超过阈值的比例")
fig.tight_layout()
plt.show()


## 本章小结

用ECDF直接展示小于等于某值的样本比例，无需选择分箱或带宽。


### 你已经掌握

- 判断累积分布图（ecdfplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 比较分位数、阈值覆盖率或不同组的完整累计分布。 |
| 数据结构 | 一列连续数值，可按类别分组。 |
| 结果解读 | 在任意X值读取累计比例，或在给定比例处读取分位值。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `stat` | proportion/count |
| `complementary` | 互补累计 |
| `hue` | 分组 |
| `weights` | 权重 |


### 需要注意

- 不理解阶梯线含义
- 组间样本量不等时比较count
- 把陡峭部分解释为时间变化


### 完成检查

- [ ] 能判断什么问题适合使用累积分布图（ecdfplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
